# ⚙️ Notebook 03 — Feature Engineering
**Step 6** | `notebooks/03_features.ipynb`

Goals:
- Create new features from existing ones
- Encode categorical variables (OHE, Label Encoding)
- Scale numerical features (StandardScaler / MinMaxScaler)
- Perform feature selection
- Test different transformations and compare

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
%matplotlib inline

os.makedirs('../models', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

## 1. Load Processed Data

In [ ]:
try:
    df = pd.read_csv('../data/processed/delivery_processed.csv')
    print('Loaded from file')
except FileNotFoundError:
    data = {
        'Distance_km': [5,22,7.93,16.42,19.52,17.44,19.03],
        'Weather': ['Clear','Rainy','Windy','Clear','Foggy','Rainy','Clear'],
        'Traffic_Level': ['Low','Medium','Low','Medium','Low','Medium','Low'],
        'Time_of_Day': ['Morning','Evening','Afternoon','Evening','Night','Afternoon','Morning'],
        'Vehicle_Type': ['Bike','Scooter','Scooter','Bike','Scooter','Scooter','Bike'],
        'Preparation_Time_min': [10,25,12,20,28,5,16],
        'Courier_Experience_yrs': [3.0,1.5,1.0,2.0,1.0,1.0,5.0],
        'Delivery_Time_min': [25,55,43,84,59,36,68],
    }
    df = pd.DataFrame(data)
    print('Loaded inline')

print('Shape:', df.shape)
df.head()

## 2. Create New Features

In [ ]:
# Feature 1: Distance × Traffic penalty
traffic_map = {'Low': 1.0, 'Medium': 1.4, 'High': 1.8}
df['Distance_Traffic'] = df['Distance_km'] * df['Traffic_Level'].map(traffic_map)
print('✅ Created: Distance_Traffic')

# Feature 2: Weather numeric factor
weather_map = {'Clear': 1.0, 'Windy': 1.1, 'Foggy': 1.2, 'Rainy': 1.3}
df['Weather_Factor'] = df['Weather'].map(weather_map)
print('✅ Created: Weather_Factor')

# Feature 3: Total delay estimate
df['Total_Delay_Est'] = (df['Distance_Traffic'] * df['Weather_Factor']
                         + df['Preparation_Time_min'])
print('✅ Created: Total_Delay_Est')

# Feature 4: Experience bucket
df['Experience_Level'] = pd.cut(
    df['Courier_Experience_yrs'],
    bins=[-0.1, 1.0, 3.0, 100],
    labels=['Junior', 'Mid', 'Senior']
)
print('✅ Created: Experience_Level')

df[['Distance_Traffic','Weather_Factor','Total_Delay_Est','Experience_Level']].head()

## 3. Correlation with Target (before encoding)

In [ ]:
num_df = df.select_dtypes(include=np.number)
corr = num_df.corr()['Delivery_Time_min'].sort_values(ascending=False)
print('Correlation with Delivery_Time_min:')
print(corr)

plt.figure(figsize=(8, 4))
corr.drop('Delivery_Time_min').plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Feature Correlation with Delivery_Time_min')
plt.ylabel('Pearson r')
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.show()

## 4. Encode Categorical Variables
### 4a. One-Hot Encoding (OHE) — recommended for tree models

In [ ]:
df_ohe = df.copy()
cat_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Experience_Level']
df_ohe = pd.get_dummies(df_ohe, columns=cat_cols, drop_first=True)
print('Shape after OHE:', df_ohe.shape)
print('New columns:', [c for c in df_ohe.columns if c not in df.columns])
df_ohe.head()

### 4b. Label Encoding — for reference / linear models

In [ ]:
df_le = df[['Weather','Traffic_Level','Time_of_Day','Vehicle_Type']].copy()
le = LabelEncoder()
for col in df_le.columns:
    df_le[col + '_encoded'] = le.fit_transform(df_le[col].astype(str))
df_le.head()

## 5. Scale Numerical Features
### 5a. StandardScaler (mean=0, std=1)

In [ ]:
num_features = ['Distance_km','Preparation_Time_min','Courier_Experience_yrs',
                'Distance_Traffic','Weather_Factor','Total_Delay_Est']

scaler_std = StandardScaler()
df_std_scaled = df_ohe.copy()
df_std_scaled[num_features] = scaler_std.fit_transform(df_ohe[num_features])
print('StandardScaler applied')
df_std_scaled[num_features].describe().round(3)

### 5b. MinMaxScaler (range 0–1)

In [ ]:
scaler_mm = MinMaxScaler()
df_mm_scaled = df_ohe.copy()
df_mm_scaled[num_features] = scaler_mm.fit_transform(df_ohe[num_features])
print('MinMaxScaler applied')
df_mm_scaled[num_features].describe().round(3)

## 6. Feature Selection — Correlation Ranking

In [ ]:
target = 'Delivery_Time_min'
corr_ohe = df_ohe.corr()[target].abs().sort_values(ascending=False)
corr_ohe = corr_ohe.drop(target)

plt.figure(figsize=(10, 5))
corr_ohe.plot(kind='bar', color='#9b59b6', edgecolor='white')
plt.title('Feature Importance (Correlation with Target) after OHE')
plt.ylabel('|Pearson r|')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

# Select top features (|r| > threshold)
threshold = 0.1
top_features = corr_ohe[corr_ohe >= threshold].index.tolist()
print(f'\nTop features (|r| >= {threshold}):')
for f in top_features:
    print(f'  {f}: {corr_ohe[f]:.3f}')

## 7. Compare Scaling Methods Visually

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

for ax, (label, data_) in zip(axes, [
    ('Original', df_ohe[num_features]),
    ('StandardScaler', df_std_scaled[num_features]),
    ('MinMaxScaler', df_mm_scaled[num_features]),
]):
    data_.boxplot(ax=ax)
    ax.set_title(label)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Scaling Method Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Save Final Feature Dataset + Scaler

In [ ]:
# Using MinMaxScaler for final dataset
final_df = df_mm_scaled.copy()
final_df.to_csv('../data/processed/delivery_features.csv', index=False)
joblib.dump(scaler_mm, '../models/scaler.pkl')

print('✅ Features saved  → data/processed/delivery_features.csv')
print('✅ Scaler saved    → models/scaler.pkl')
print('Final shape:', final_df.shape)
final_df.head()